# 01 - Data Cleaning and Preparation

**Kenya Food Price Inflation Tracker**

## Objective
Load and clean the WFP Kenya food price dataset, preparing it for analysis.

## Steps
1. Load raw WFP data
2. Create date column from year/month
3. Handle missing values and outliers
4. Filter to core staple commodities
5. Create cleaned datasets

## Outputs
- `data/clean/wfp_kenya_clean.csv`
- `data/clean/wfp_core_staples.csv`
- `data/clean/wfp_monthly_avg.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

print('✓ Libraries imported')

## 1. Load Raw Data

In [ ]:
# Load data
try:
    df = pd.read_csv('../data/raw/wfp_food_prices_kenya_full.csv')
except FileNotFoundError:
    df = pd.read_csv('data/raw/wfp_food_prices_kenya_full.csv')

print(f'Dataset shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nDate range: {df["mp_year"].min()}-{df["mp_month"].min():02d} to {df["mp_year"].max()}-{df["mp_month"].max():02d}')
df.head()

In [ ]:
# Check data types and missing values
print('\n=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0])

print('\n=== Data Types ===')
print(df.dtypes)

## 2. Create Date Column

In [ ]:
# Create date column from year and month
df['date'] = pd.to_datetime({'year': df['mp_year'], 'month': df['mp_month'], 'day': 1})
df = df.sort_values('date').reset_index(drop=True)

print(f'✓ Date column created')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
df[['date', 'cm_name', 'mp_price', 'adm1_name', 'mkt_name']].head()

## 3. Data Cleaning

In [ ]:
# Drop rows with missing prices
print(f'Rows before: {len(df)}')
df = df.dropna(subset=['mp_price'])
df = df[df['mp_price'] > 0]
print(f'Rows after: {len(df)}')
print(f'Removed: {len(df) - len(df)} invalid price records')

In [ ]:
# Remove outliers using IQR method (per commodity)
def remove_outliers_iqr(group, column='mp_price', multiplier=3.0):
    Q1 = group[column].quantile(0.25)
    Q3 = group[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    return group[(group[column] >= lower) & (group[column] <= upper)]

print(f'Before outlier removal: {len(df):,} rows')
df_clean = df.groupby('cm_name', group_keys=False).apply(remove_outliers_iqr)
print(f'After outlier removal: {len(df_clean):,} rows')
print(f'Removed: {len(df) - len(df_clean):,} outlier records')

## 4. Focus on Core Staple Commodities

In [ ]:
# Check available commodities
print('All commodities:')
print(df_clean['cm_name'].value_counts())

In [ ]:
# Define core staples (using ACTUAL commodity names from data)
# Look for: Maize, Beans, Rice, Oil, Sugar, Milk
core_staples = [
    'Maize - Retail',
    'Maize - Wholesale', 
    'Maize (white) - Retail',
    'Beans (dry) - Retail',
    'Beans - Wholesale',
    'Rice - Retail',
    'Oil (vegetable) - Retail',
    'Sugar - Retail',
    'Milk (cow, pasteurized) - Retail'
]

# Filter to commodities that actually exist in data
available_staples = [c for c in core_staples if c in df_clean['cm_name'].values]
print(f'\nCore staples available: {len(available_staples)}')
print(available_staples)

df_staples = df_clean[df_clean['cm_name'].isin(available_staples)].copy()
print(f'\nStaples dataset: {df_staples.shape}')
print(df_staples['cm_name'].value_counts())

## 5. Create Monthly Aggregated Data

In [ ]:
# Aggregate to monthly averages (by commodity, region, and market)
df_monthly = df_staples.groupby(
    ['date', 'cm_name', 'adm1_name', 'mkt_name']
).agg({
    'mp_price': ['mean', 'std', 'count'],
    'mkt_id': 'first',
    'cur_name': 'first'
}).reset_index()

# Flatten column names
df_monthly.columns = ['date', 'commodity', 'region', 'market', 
                       'price_mean', 'price_std', 'obs_count',
                       'market_id', 'currency']

print(f'Monthly dataset: {df_monthly.shape}')
df_monthly.head(10)

## 6. Save Cleaned Datasets

In [ ]:
# Create output directory
Path('../data/clean').mkdir(parents=True, exist_ok=True)

# Save datasets
df_clean.to_csv('../data/clean/wfp_kenya_clean.csv', index=False)
print(f'✓ Saved: wfp_kenya_clean.csv ({len(df_clean):,} rows)')

df_staples.to_csv('../data/clean/wfp_core_staples.csv', index=False)
print(f'✓ Saved: wfp_core_staples.csv ({len(df_staples):,} rows)')

df_monthly.to_csv('../data/clean/wfp_monthly_avg.csv', index=False)
print(f'✓ Saved: wfp_monthly_avg.csv ({len(df_monthly):,} rows)')

## 7. Summary

In [ ]:
print('=== DATA CLEANING SUMMARY ===')
print(f'Original records: {len(df):,}')
print(f'After cleaning: {len(df_clean):,}')
print(f'Core staples: {len(df_staples):,}')
print(f'Monthly aggregated: {len(df_monthly):,}')
print(f'\nDate range: {df_staples["date"].min()} to {df_staples["date"].max()}')
print(f'Commodities: {df_staples["cm_name"].nunique()}')
print(f'Markets: {df_staples["mkt_name"].nunique()}')
print(f'Regions: {df_staples["adm1_name"].nunique()}')
print('\n✅ Data cleaning complete!')